<a href="https://colab.research.google.com/github/Aerospace87/ML-projects/blob/main/ExercisesOnTrasformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
!pip install diffusers["torch"] transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 83.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 70.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 26.5 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
model_type = "Qwen/Qwen2-0.5B"
model = AutoModelForCausalLM.from_pretrained(model_type)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


In [ ]:
prompt = """\
  Translate English to Spanish:
  English: I do not speak Spanish.
  Spanish: No hablo español.
  English: """
tokenizer = AutoTokenizer.from_pretrained(model_type)
input_ids = tokenizer(prompt, return_tensors="pt").input_ids

In [ ]:
for t in input_ids:
  print(t, '\t:', tokenizer.decode(t))

tensor([  220, 37740,  6364,   311, 15154,   510,   220,  6364,    25,   358,
          653,   537,  6468, 15154,   624,   220, 15154,    25,  2308,  6055,
          385, 69888,   624,   220,  6364,    25,  3496,   498,  2937,  4894,
          220, 15154,    25, 48813,    39, 14300, 60394,  4894,   220,  6364,
           25, 10967,   374,   264,  1661, 10729,  5267,   220, 15154,    25,
        28286,    35,  1794, 42341, 17669,   650, 53713,  7544,  4942,  5267,
          220,  6364,    25,  3555, 12026,   653,   498,   614,  2500,  5267,
          220, 15154,    25, 28286, 65706, 14132, 12430, 23332, 69620,  5267,
          220,  6364,    25,   358,  1075, 22174,   198,   220, 15154,    25]) 	:   Translate English to Spanish:
  English: I do not speak Spanish.
  Spanish: No hablo español.
  English: See you later!
  Spanish: ¡Hasta luego!
  English: Where is a good restaurant?
  Spanish: ¿Dónde hay un buen restaurante?
  English: What rooms do you have available?
  Spanish: ¿Qué habi

In [ ]:
outputs_ids = model(input_ids)
outputs_ids.logits.shape

torch.Size([1, 90, 151936])

In [ ]:
final_logits = outputs_ids.logits[0, -1]
probabilities = final_logits.softmax(dim=0)
most_probable_word = final_logits.argmax()

In [ ]:
tokenizer.decode(most_probable_word)

' Me'

In [ ]:
top10_logits = torch.topk(final_logits, 10)
for index in top10_logits.indices:
  print(tokenizer.decode(index))

 Me
 A
 I
 Yo
 En
 me
 Mi
 Am
 M
 S


In [ ]:
top_10_prob = torch.topk(probabilities, 10)

In [ ]:
for index, prob in zip(top_10_prob.indices, top_10_prob.values):
  print(f"{tokenizer.decode(index)}: {100. * prob:.2f}%")

 Me: 72.58%
 A: 8.41%
 I: 1.42%
 Yo: 1.05%
 En: 0.88%
 me: 0.82%
 Mi: 0.70%
 Am: 0.59%
 M: 0.54%
 S: 0.54%


In [ ]:
output_ids= model.generate(input_ids, max_new_tokens=20)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


In [ ]:
decoded_text = tokenizer.decode(output_ids[0])
print(decoded_text)

  Translate English to Spanish:
  English: I do not speak Spanish.
  Spanish: No hablo español.
  English: See you later!
  Spanish: ¡Hasta luego!
  English: Where is a good restaurant?
  Spanish: ¿Dónde hay un buen restaurante?
  English: What rooms do you have available?
  Spanish: ¿Qué habitaciones tiene disponibles?
  English: I like soccer
  Spanish: Me gusta el fútbol
  English: I like to play tennis
  Spanish:


In [ ]:
beam_output_ids= model.generate(
    input_ids,
    num_beams = 5,
    repetition_penalty = 2.0 ,
    temperature = 0.4,
    max_new_tokens=45,
    do_sample =True,
    top_p = 0.94,
    top_k = 0)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


In [ ]:
beam_decoded_text = tokenizer.decode(beam_output_ids[0])

In [ ]:
print(beam_decoded_text)

  Translate English to Spanish:
  English: I do not speak Spanish.
  Spanish: No hablo español.
  English: See you later!
  Spanish: ¡Hasta luego!
  English: Where is a good restaurant?
  Spanish: ¿Dónde hay un buen restaurante?
  English: What rooms do you have available?
  Spanish: ¿Qué habitaciones tiene disponibles?
  English: I like soccer
  Spanish: Me gusta el fútbol
  English: How are you today?
  Spanish: ¿Cómo estás hoy?
  English: Would you like to come with me?
  Spanish: ¿Quieres venir conm


## Pipelines

In [ ]:
from transformers import pipeline

In [ ]:
fill_masker = pipeline("fill-mask", model="bert-base-uncased")
fill_masker("The [MASK] is made of milk.")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Device set to use cpu


[{'score': 0.19546712934970856,
  'token': 9841,
  'token_str': 'dish',
  'sequence': 'the dish is made of milk.'},
 {'score': 0.12907519936561584,
  'token': 8808,
  'token_str': 'cheese',
  'sequence': 'the cheese is made of milk.'},
 {'score': 0.1059068813920021,
  'token': 6501,
  'token_str': 'milk',
  'sequence': 'the milk is made of milk.'},
 {'score': 0.04112080857157707,
  'token': 4392,
  'token_str': 'drink',
  'sequence': 'the drink is made of milk.'},
 {'score': 0.03712376952171326,
  'token': 7852,
  'token_str': 'bread',
  'sequence': 'the bread is made of milk.'}]

In [ ]:
classifier = pipeline("text-classification",\
                      model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")

classifier("This movie is disgustingly good !")


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Device set to use cpu


[{'label': 'POSITIVE', 'score': 0.9998536109924316}]

### Create model generation


In [ ]:
model_type = "Qwen/Qwen2-0.5B"
model = AutoModelForCausalLM.from_pretrained(model_type)
prompt = """I love """
tokenizer = AutoTokenizer.from_pretrained(model_type)
input_ids = tokenizer(prompt, return_tensors="pt").input_ids

In [ ]:
def generate(model, tokenizer, input_ids, max_length=50, do_sample=False, top_k=None):

  """Generate a sequence without using
    model.generate()
    Args:
    model: The model to use for generation.
    tokenizer: The tokenizer to use for generation.
    input_ids: The input IDs
    max_length: The maximum length of the sequence.
    do_sample: Whether to use sampling.
    top_k: The number of tokens to sample from.
  """
  # Write your code here
  # Begin by the simplest approach, greedy decoding.
  # Then add sampling and finally top-k sampling.

  if not do_sample:
    while input_ids.size()[1] <= max_length:
        outputs_ids = model(input_ids)
        final_logits = outputs_ids.logits[0, -1]
        idx_most_probable_word = final_logits.argmax()
        input_size = input_ids.size()[1]
        input_ids = torch.cat((input_ids, torch.tensor([[idx_most_probable_word]])), dim=1)

  elif do_sample and (top_k == None):
      while input_ids.size()[1] <= max_length:
        outputs_ids = model(input_ids)
        final_logits = outputs_ids.logits[0, -1]
        probabilities = final_logits.softmax(dim=0)
        idx_sampled_from_proba_distribution = torch.multinomial(probabilities, num_samples = 1)
        input_size = input_ids.size()[1]
        input_ids = torch.cat((input_ids, torch.tensor([[idx_sampled_from_proba_distribution]])), dim=1)

  elif do_sample and isinstance(top_k, int):
        while input_ids.size()[1] <= max_length:
          outputs_ids = model(input_ids)
          final_logits = outputs_ids.logits[0, -1]
          probabilities = final_logits.softmax(dim=0)
          # sorting probabilities decending
          sorted, indices = torch.sort(probabilities, descending=True, axis = 0)
          # Choosing the highest top 5 probabilities and related indices
          top_k_prob = sorted[:top_k]
          top_indices = indices[:top_k]
          # Renormalizing between 5 values
          top_k_prob = top_k_prob / torch.sum(top_k_prob)

          # idx of the position in top_indices
          idx = torch.multinomial(top_k_prob, num_samples = 1)
          # right idx related to token
          idx = top_indices[idx]

          input_size = input_ids.size()[1]
          input_ids = torch.cat((input_ids, torch.tensor([[idx]])), dim=1)
  else:
    raise ValueError('Not supported!')

  print(tokenizer.decode(input_ids[0]))

In [ ]:
generate(model, tokenizer, input_ids, max_length=40, do_sample = True, top_k=5)

I love 100% of my life.
I am a woman of faith, I am a woman of hope, a woman of love, a woman of peace, and a woman of joy.


## Summary of paragraph

In [2]:
from transformers import pipeline

In [3]:
summarizer = pipeline("summarization")

No model was supplied, defaulted to sshleifer/distilbart-cnn-12-6 and revision a4f8f3e (https://huggingface.co/sshleifer/distilbart-cnn-12-6).
Using a pipeline without specifying a model name and revision in production is not recommended.
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.80k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

Device set to use cuda:0


In [4]:
paragraph = """Cancer is a group of diseases involving abnormal cell growth with\n
the potential to invade or spread to other parts of the body.\n
These contrast with benign tumors, which do not spread.\n
Possible signs and symptoms include a lump, abnormal bleeding, prolonged cough,\n
unexplained weight loss, and a change in bowel movements.\n
While these symptoms may indicate cancer, they can also have other causes.\n
Over 100 types of cancers affect humans."""

In [5]:
summarizer(paragraph, min_length=10, max_length=60)

[{'summary_text': ' Cancer is a group of diseases involving abnormal cell growth with the potential to invade or spread to other parts of the body . These contrast with benign tumors, which do not spread . Symptoms include a lump, abnormal bleeding, prolonged cough and prolonged cough .'}]

In [6]:
zero_shot_prompt = f"""Summarize the following paragraph:
paragraph:{paragraph}
summary:
"""

In [7]:
summarizer(zero_shot_prompt,min_length=10, max_length=90)

[{'summary_text': ' Cancer is a group of diseases involving abnormal cell growth with the potential to invade or spread to other parts of the body . Symptoms include a lump, abnormal bleeding, prolonged cough and prolonged cough .'}]

In [13]:
one_shot_prompt = f"""Summarize the second paragraph:

first paragraph:A bicycle, also called a pedal cycle, bike, push-bike or cycle,
      is a human-powered or motor-assisted, pedal-driven, single-track vehicle,
      with two wheels attached to a frame, one behind the other.
      A bicycle rider is called a cyclist, or bicyclist.

summary:A bicycle is a mode of transport used by people. The person pedals and a
        couple of wheels rotates.

second paragraph:{paragraph}
summary:
"""

In [14]:
summarizer(one_shot_prompt,min_length=10, max_length=90)

[{'summary_text': ' A bicycle, also called a pedal cycle, bike, push-bike or cycle, is a human-powered or motor-assisted, pedal-driven, single-track vehicle, with two wheels attached to a frame, one behind the other .'}]

## Sentiment Analysis

In [16]:
sentiment_analyzer = pipeline('sentiment-analysis', model="distilbert-base-uncased-finetuned-sst-2-english")

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Device set to use cuda:0


In [17]:
review = 'This move is incredible!'

In [18]:
zero_shot_prompt =f"""Question: Is the following review positive or negative about the movie?
Review: {review}
Answer:"""

In [23]:
sentiment_analyzer(zero_shot_prompt)

[{'label': 'NEGATIVE', 'score': 0.9997689127922058}]

In [21]:
review = 'This move is without an interesting plot!'

In [22]:
zero_shot_prompt =f"""Question: Is the following review positive or negative about the movie?
Review: {review}
Answer:"""

In [24]:
sentiment_analyzer(zero_shot_prompt)

[{'label': 'NEGATIVE', 'score': 0.9997689127922058}]

## Semantic Search